# Practical: Molecular dynamics simulations of a small protein domain

By:  [ Camilo Aponte-Santamaría, MPI for Polymer Research, Mainz Germany](https://mptg-cbp.github.io/caas/)

**Acknowledgement:** This tutorial is based on the [<ins>tutorial written by Bert de Groot</ins>](http://www3.mpibpc.mpg.de/groups/de_groot/compbio1/p2/index.html).

---

**Assignment 2 extension (this notebook is the main working notebook for the assignment):**
the original practical below simulates the B1 domain of protein G for 20 ps with a single force
field (OPLS-AA/L + SPC water). For Assignment 2 we extend it to a **2 ns, three-force-field
comparison** — OPLS-AA/L, GROMOS96 43a1 (united-atom), and GRAPPA (machine-learned) — following
`docs/assignment/assignment2_description.pdf` and the scoping decisions in
`docs/assignment/assignment2_method_clarifications.tex`.

- **Parts 1–A** below are unchanged background/preparation (one shared starting structure for all
  three force fields, no simulation run yet — safe to read once).
- From **Part B** onward, the originally single, hard-coded, interactive `gmx` walkthrough is
  replaced by one **parametrized, non-interactive pipeline** driven by a `FORCE_FIELDS` dict and
  run in a loop over three separate working directories (`run_oplsaa/`, `run_gromos/`,
  `run_grappa/`) — no copy-pasted cells per force field, and nothing depends on manually typing
  answers to `gmx` prompts.
- **Part D (Analysis)** now produces **overlay** plots comparing the three force fields directly
  (RMSD/Rg for both whole-protein and backbone/Cα selections, RMSF, potential energy, SASA), plus
  the particle-count/performance comparison the assignment asks for in Q1.
- **Part E** is new: a discussion scaffold mapped to the assignment's 7 graded questions, and a
  suggested report structure.

This notebook installs GROMACS and needs several hours of wall-clock time for the three 2 ns
runs — it is meant to be **run in Google Colab**, not executed in this repository/sandbox. No
simulation output exists in this repo yet.


##1) Introduction

In this practical we will carry out a molecular dynamics (MD) simulation of a biological macromolecule: a small protein.

Molecular dynamics (MD) simulations and related computational tools have nowadays become essential tools for the study of biomolecular systems. Over the last decade, it has been successfully
elucidated key biological processes such as enzyme catalysis, ribosomal translocation, DNA repair, membrane permeation, protein folding, macromolecular crowding and viral assembly, among many
others. The use of this technique enables monitoring the dynamics of biomolecules in an unprecedentedly broad spatio-temporal range, from pico- to milli-seconds and from thousands to hundreds of millions of atoms. Moreover, recent advances in computing power, together with optimized algorithms, machine learning strategies, and new sampling methodologies, are pushing these boundaries further, while increasing the precision and accuracy. Simulations do not only help interpret experimental data but also complement experimental techniques such as Cryo-EM, FRET, SAXS, or NMR to reveal key structure, dynamic, and energetic information of biomolecules.

Today, we will simulate the dynamics of a small, typical protein domain: the B1 domain of protein G. B1 is one of the domains of protein G, a member of an important class of proteins, which form IgG binding receptors on the surface of certain Staphylococcal and Streptococcal strains. These proteins allow the pathogenic bacterium to evade the host immune response by coating the invading bacteria with host antibodies, thereby contributing significantly to the pathogenicity of these bacteria.

After carrying out the simulation, we will proceed analyzying its outcome illustrating several of the concepts we saw in the lecture.

**For Assignment 2**, we repeat this whole workflow with three different force fields (OPLS-AA/L,
GROMOS96 43a1, GRAPPA) and 2 ns of simulation time each, then compare stability (RMSD, radius of
gyration), flexibility (RMSF), energetics, and surface behavior (SASA) between them.


## 2) Running This Notebook

We will need to install:

*   **[GROMACS:](https://www.gromacs.org/)** MD engine. GROMACS also provides a set of `gmx` tools to analyze the trajectories. We will use those.
*   **[PyMol](https://pymol.org/2/):** for the visualization of the MD trajectories. **Please install this program in your computer.**

Analysis and plotting are done with `numpy`, `pandas`, and `matplotlib` — no additional Python
packages are required.


In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y gromacs
# pymol: install it in your laptop


### Import libraries and shared helper functions

In [ ]:
import re
import shutil
import subprocess
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
INPUTS_DIR = BASE_DIR / "inputs"
INPUTS_DIR.mkdir(exist_ok=True)


The helper functions below are the "clean up" of the original tutorial's approach of running one
`!gmx ...` shell command per cell and answering its interactive prompts by hand. Instead:

- `run()` executes a shell command and, if the `gmx` tool needs an interactive group selection
  (e.g. "Select a group for RMSD calculation"), feeds it via `input_text` instead of a manual
  prompt — this is what makes the pipeline safe to loop over three force fields unattended.
- `download()` is a small `wget` replacement used for the PDB structure, the `.mdp` files, and the
  GRAPPA topology.
- `load_xvg()` reads GROMACS' `.xvg` output format (same as the original tutorial's
  `np.loadtxt(..., comments=['#','@'])`, wrapped for reuse and with optional ps→ns conversion,
  since 2 ns trajectories are easier to read in ns than in ps).
- `set_mdp_param()` rewrites a single `key = value` line in an `.mdp` file — used to change
  `nsteps` for the 2 ns extension without hand-editing the file.
- `parse_performance()` / `parse_atom_counts()` extract the numbers needed for Q1 (performance,
  particle counts) directly from `md.log` / `topol.top`, instead of reading them off the screen.


In [ ]:
def run(cmd, input_text=None, cwd=None, check=True):
    """Run a shell command non-interactively, optionally feeding stdin (e.g. gmx group
    selections). Prints a trimmed tail of stdout/stderr so long GROMACS logs don't flood
    the notebook, but failures are always visible."""
    result = subprocess.run(
        cmd, shell=True, cwd=cwd, input=input_text, text=True,
        capture_output=True,
    )
    tail = lambda s, n=2500: s[-n:] if s else s
    print(f"$ {cmd}  (cwd={cwd or '.'})")
    if result.stdout:
        print(tail(result.stdout))
    if check and result.returncode != 0:
        print(tail(result.stderr))
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result


def download(url, dest_dir, filename=None):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / (filename or url.rsplit("/", 1)[-1])
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest


def load_xvg(path, to_ns=False):
    """Load a GROMACS .xvg file, skipping comment/annotation lines. Returns an array
    with shape (n_columns, n_rows); optionally converts column 0 (time, ps) to ns."""
    data = np.loadtxt(path, comments=["#", "@"])
    data = np.atleast_2d(data).T
    if to_ns:
        data[0] = data[0] / 1000.0
    return data


def set_mdp_param(mdp_path, key, value, out_path=None):
    """Rewrite (or append) a single `key = value` line in a .mdp file."""
    mdp_path = Path(mdp_path)
    out_path = Path(out_path) if out_path else mdp_path
    lines = mdp_path.read_text().splitlines()
    pattern = re.compile(rf"^(\s*{re.escape(key)}\s*=).*")
    replaced = False
    for i, line in enumerate(lines):
        if pattern.match(line):
            lines[i] = f"{key:<24} = {value}"
            replaced = True
            break
    if not replaced:
        lines.append(f"{key:<24} = {value}")
    out_path.write_text("\n".join(lines) + "\n")
    return out_path


def parse_performance(log_path):
    """Extract ns/day and hour/ns from the 'Performance:' line of an md.log file."""
    text = Path(log_path).read_text()
    m = re.search(r"Performance:\s*([\d.]+)\s+([\d.]+)", text)
    if not m:
        return {"ns_per_day": np.nan, "hour_per_ns": np.nan}
    return {"ns_per_day": float(m.group(1)), "hour_per_ns": float(m.group(2))}


def parse_atom_counts(topol_path, protein_moltype_prefix="Protein"):
    """Parse a GROMACS topology (.top) to get (a) the number of atoms per moleculetype,
    from each [ atoms ] block, and (b) the number of copies of each molecule from the
    final [ molecules ] section, then combine both into total system / protein-only
    atom counts. Works for topol.top written directly by pdb2gmx (single-file topology,
    as in this practical) as well as the supplied GRAPPA topology."""
    text = Path(topol_path).read_text()
    blocks = re.split(r"\[\s*moleculetype\s*\]", text)[1:]
    atoms_per_moltype = {}
    for block in blocks:
        name_match = re.search(r"^\s*(\S+)", block, re.MULTILINE)
        moltype = name_match.group(1)
        atoms_section = re.search(
            r"\[\s*atoms\s*\](.*?)(?:\n\s*\[|\Z)", block, re.DOTALL
        )
        n_atoms = 0
        if atoms_section:
            for line in atoms_section.group(1).splitlines():
                line = line.split(";")[0].strip()
                if line:
                    n_atoms += 1
        atoms_per_moltype[moltype] = n_atoms

    molecules_section = re.search(r"\[\s*molecules\s*\](.*)", text, re.DOTALL)
    total_atoms = 0
    protein_atoms = 0
    counts = {}
    matched_protein_moltypes = []
    if molecules_section:
        for line in molecules_section.group(1).splitlines():
            line = line.split(";")[0].strip()
            if not line:
                continue
            name, n_str = line.split()
            n = int(n_str)
            counts[name] = n
            n_atoms_here = atoms_per_moltype.get(name, 0) * n
            total_atoms += n_atoms_here
            if name.startswith(protein_moltype_prefix):
                protein_atoms += n_atoms_here
                matched_protein_moltypes.append(name)
    # Defensive: if nothing matched the protein prefix (e.g. the supplied GRAPPA topology names
    # its protein moleculetype differently), the protein-atom count would silently be 0 and every
    # Q1 comparison would be wrong. Surface that here instead of hiding it.
    if not matched_protein_moltypes:
        print(f"WARNING: parse_atom_counts found no moleculetype starting with "
              f"'{protein_moltype_prefix}' in {topol_path}. Molecule names seen: {list(counts)}. "
              f"Protein-atom count is 0 - check the protein_moltype_prefix against this topology.")
    return {
        "n_atoms_protein": protein_atoms,
        "n_atoms_total": total_atoms,
        "molecule_counts": counts,
        "atoms_per_moltype": atoms_per_moltype,
        "matched_protein_moltypes": matched_protein_moltypes,
    }


In [ ]:
!gmx -h


## A) Preparation

<BR CLEAR=”left” />
<img align=right src="https://drive.google.com/uc?export=view&id=1PpM_OAUoVWWLqxhzi6ZUqiE8BE_8cpam" width=400>

The simulations will be carried out with the [GROMACS](https://www.gromacs.org/) simulation package. On the GROMACS homepage you can find both the software and documentation (online reference and manual). To run a simulation, three input files are usually required:

1.   **a structure** file, containing the initial atomic coordinates of the system to be simulated (**pdb or gro format**)
2.   **a "molecular topology"**, containing the atomic simulation parameters (force-field) and a description of the bonds, bond angles, etc (if any) of the simulation system (**top format**)
3.   **the simulation parameters**: type of simulation, number of steps, simulation temperature, etc. (**mdp format**)

We will now follow a standard protocol to run a typical MD simulation of a protein in a box of water in GROMACS. The individual steps are summarized in the [flowchart](https://mptg-cbp.github.io/teaching/tutorials/protein/flow.png) shown at the right.

This structure and protocol are **shared by all three force fields** in the Assignment 2
comparison — only the force field/water model choice in `pdb2gmx` (Part B) differs between them.


### Initial conformation
Before a simulation can be started, an initial structure of the protein is required. Fortunately, the structure of the B1 domain of protein G has been solved experimentally, both by x-ray crystallography and NMR. Experimentally solved protein structures are collected and distributed by the [Protein Data Bank (PDB)](https://www.rcsb.org/). Please open this link in a new browser window and enter **"protein G" "B1"** in the search field. Several entries in the PDB should match this query. We will choose the x-ray structure with the highest resolution (**entry 1PGB**) for this study.

To download the structure, click on the link `1PGB`, and then, under `Download Files`, click the secondary button on `Legacy PDB format` and then select `Copy link`.

We download the structure with the `download()` helper (a small `wget` wrapper) into a shared
`inputs/` directory, so it can be reused by all three force-field pipelines below.


In [ ]:
PDB_URL = "https://files.rcsb.org/download/1PGB.pdb"
pdb_path = download(PDB_URL, INPUTS_DIR)
pdb_path


To have a look at the contents of the downloaded PDB file

In [ ]:
with open(pdb_path) as file:
  line=file.read()
  print(line)


The file starts with general information about the protein, about the structure, and about the experimental techniques used to determine the structure, as well as literature references where the structure is described in detail. The data file contains the atomic coordinates of our protein structure with one line per atom (lines starting with `ATOM` and `HETATM`).


#### PyMol visualization
<BR CLEAR=”left” />
<img align=right src="https://mptg-cbp.github.io/teaching/tutorials/protein/snap_1pgb.png" width=300>

Let us visualize the structure with `PyMol`.

*   Download `inputs/1PGB.pdb` to your local hard disk.
*   Open `PyMoL` in your computer.
*   In the PyMoL `File` menu, click `open` and select the downloaded `1PGB.pdb` file.
*   PyMoL displays the cartoon representation together with non-bonded crystallographic water molecules. Under the "Show" and "Hide" menus ("S" and "H" at the top-right of the "PyMOL Viewer" window), also try other representations such as "wire", "sticks", "spheres", and "surface". Try different views by moving the mouse over the molecule viewer with the left mouse button pressed, and zoom with the right mouse button pressed.
*   Exit pymol and continue in Google colab.


<font color=red size=+1> **QUESTION:** </font> Why do we start our MD simulations from the experimental determined 3D structure? Isn't it enough to know the proteins amino acid sequence?


## B) Setup

The original tutorial prepares topology, box, solvent, and ions for a **single** force field
(OPLS-AA/L + SPC), typing answers into `gmx pdb2gmx`'s interactive prompt. For Assignment 2 we
need to repeat this exact protocol for three force fields, so instead we:

1. define one `FORCE_FIELDS` configuration (force field name, `pdb2gmx` flags, working directory),
2. wrap the whole per-force-field protocol (`pdb2gmx` → `editconf` → `solvate` → `genion` → energy
   minimization) in a single `setup_system()` function,
3. call it once per entry in `FORCE_FIELDS`, each in its **own subdirectory**, matching the
   assignment's recommendation to *"carry out the tutorial in separate directories so files are
   not overwritten."*

### add hydrogen atoms and topology generation

We now prepare the protein structure to be simulated in GROMACS. Although we now have a starting structure for our protein, one might have noticed that hydrogen atoms (which would appear white) are still missing from the structure. This is because hydrogen atoms contain too few electrons to be observed by x-ray crystallography at moderate resolutions. Also, GROMACS requires a molecular description (or topology) of the molecules to be simulated before we can start, containing information on e.g. which atoms are covalently bonded and other physical information. Both the generation of hydrogen atoms and writing of the topology can be done with the GROMACS program `pdb2gmx`.

Rather than being prompted interactively for the force field and water model, `pdb2gmx` is called
with explicit `-ff`/`-water` flags:

| Force field | `-ff` | `-water` | Notes |
|---|---|---|---|
| **OPLS-AA** | `oplsaa` | `spc` | baseline, same as the original practical |
| **GROMOS**  | `gromos43a1` | `spc` | united-atom (some CH₂/CH₃ groups become single particles) |
| **GRAPPA**  | `amber99sb-ildn` (initial topology only) | `tip3p` | topology is then replaced by the assignment-supplied `topology_grappa.top` (machine-learned parameters) — see the `use_grappa_topology` handling below |


In [ ]:
FORCE_FIELDS = {
    "OPLS-AA": dict(
        workdir="run_oplsaa",
        pdb2gmx_ff="oplsaa",
        water="spc",
        use_grappa_topology=False,
        label="OPLS-AA/L + SPC (baseline)",
    ),
    "GROMOS": dict(
        workdir="run_gromos",
        pdb2gmx_ff="gromos43a1",
        water="spc",
        use_grappa_topology=False,
        label="GROMOS96 43a1 + SPC (united-atom)",
    ),
    "GRAPPA": dict(
        workdir="run_grappa",
        pdb2gmx_ff="amber99sb-ildn",  # initial topology only; replaced by topology_grappa.top below
        water="tip3p",
        use_grappa_topology=True,
        label="AMBER99SB-ILDN/TIP3P topology converted to GRAPPA (machine-learned)",
    ),
}
GRAPPA_TOP_URL = "https://mptg-cbp.github.io/teaching/tutorials/protein/topology_grappa.top"


**The topology file** written by `pdb2gmx` is called `"topol.top"` (one copy per force-field
working directory below). It contains:

*   all the atoms (with masses, charges)
*   bonds (covalent bonds connecting the atoms)
*   angles
*   dihedral angles
*   etc.

Near the very end of the topology (in the `"[molecules]"` section) there is a summary of the
simulation system, including the protein and the crystallographic water molecules. It thus
contains all the physical information about all interactions between the atoms of the protein
(bonds, angles, torsion angles, Lennard-Jones interactions and electrostatic interactions) — this
is exactly what `parse_atom_counts()` above reads to answer the Q1 particle-count question.

### Solvate the protein, add ions, minimize

The next steps in setting up each simulation system are: define a simulation box (rectangular,
edges at least 0.7 nm from the protein surface), solvate it, neutralize the net charge with ions,
and run an energy minimization to remove bad contacts. This mirrors the original tutorial's
`editconf` → `solvate` → `genion` → `grompp`/`mdrun` (em) steps, but:

- **ion neutralization** uses `genion -neutral` (auto-computed ion count) instead of a hard-coded
  `-np 4`, so *total charge* neutralization stays consistent across force fields even if exact
  atom-level bookkeeping differs slightly (clarification Q3);
- **the GRAPPA topology substitution** happens right after `pdb2gmx`: the AMBER99SB-ILDN/TIP3P
  `topol.top` that `pdb2gmx` wrote is overwritten with the downloaded `topology_grappa.top`, so
  every later command that takes `-p topol.top` transparently uses the GRAPPA parameters — matching
  the assignment's instruction to substitute the file for *"every gmx command that takes a .top
  file as input"*, while keeping one uniform pipeline for all three force fields;
- **water box coordinates**: `solvate` uses the generic pre-equilibrated `spc216.gro` box for all
  three force fields (a standard simplification also used for TIP3P in most GROMACS tutorials,
  since it only provides initial 3-site-water coordinates — the physics comes from the water model
  block written into the topology, not from this coordinate file).


In [ ]:
def setup_system(name, cfg):
    """pdb2gmx -> editconf -> solvate -> genion -> energy minimization, for one force field."""
    workdir = BASE_DIR / cfg["workdir"]
    workdir.mkdir(exist_ok=True)
    em_mdp_path = download("https://mptg-cbp.github.io/teaching/tutorials/protein/em.mdp", workdir)
    shutil.copy(pdb_path, workdir / "1PGB.pdb")

    print(f"\n=== {name}: {cfg['label']} ===")
    # TODO/CAUTION (runtime only): 1PGB from the PDB has no hydrogens, so pdb2gmx adds them per
    # force field. If a GROMOS/GRAPPA run aborts on pre-existing/duplicate H atoms, add `-ignh` to
    # the pdb2gmx call below. Not added by default so the protocol stays identical across FFs.
    run(
        f"gmx pdb2gmx -f 1PGB.pdb -o conf.pdb -ff {cfg['pdb2gmx_ff']} -water {cfg['water']}",
        cwd=workdir,
    )
    if cfg["use_grappa_topology"]:
        grappa_top = download(GRAPPA_TOP_URL, workdir, filename="topology_grappa.top")
        shutil.copy(grappa_top, workdir / "topol.top")

    run("gmx editconf -f conf.pdb -o box.pdb -d 0.7", cwd=workdir)
    run("gmx solvate -cp box.pdb -cs spc216 -o water.pdb -p topol.top", cwd=workdir)

    # TODO/CAUTION (runtime only): -maxwarn 2 mirrors the original tutorial. GROMOS/GRAPPA grompp
    # may emit different notes; if grompp aborts on a warning, read it first and only then raise
    # -maxwarn here (and in run_production_md). Do NOT increase it blindly - it can mask real
    # setup errors.
    run("gmx grompp -f em.mdp -c water.pdb -p topol.top -o em_ion.tpr -maxwarn 2", cwd=workdir)
    run(
        "gmx genion -s em_ion.tpr -o ions.pdb -p topol.top -pname NA -nname CL -neutral",
        input_text="SOL\n",
        cwd=workdir,
    )

    run("gmx grompp -f em.mdp -c ions.pdb -p topol.top -o em.tpr -maxwarn 2", cwd=workdir)
    run("gmx mdrun -v -s em.tpr -c em.pdb -deffnm em", cwd=workdir)

    return workdir


systems = {}
for name, cfg in FORCE_FIELDS.items():
    systems[name] = setup_system(name, cfg)


View the resulting `conf.pdb`, `box.pdb`, `water.pdb`, and `ions.pdb` in each working directory
with `PyMoL` as in the original tutorial (protein in licorice/stick, simulation box with
`show cell`, ions colored purple with `color purple, resn NA` / `show spheres, resn NA`) — useful
as a sanity check per force field, but not required for every run once you've confirmed the
pipeline works once.


##C) Simulation

Now we have all that is required to start the dynamics for each force field. The original
tutorial downloads a single `md.mdp` and runs 20 ps (`nsteps` in the shipped file). For Assignment
2 we **extend the production run from 20 ps to 2 ns**, keeping `dt = 0.002 ps` fixed (so
`nsteps = 2000 ps / 0.002 ps`), and keep the thermostat/constraint settings exactly as in the
original file for consistency across all three force fields (clarifications Q4/Q5).

A `TEST_MODE` switch first runs every force field through the full pipeline with a short
simulation (a few ps) to catch setup mistakes cheaply, before committing to the full, expensive
2 ns runs (~1.5 h per force field on a standard 4-core machine, ~4.5 h total per the assignment's
own estimate).


In [ ]:
md_mdp_path = download("https://mptg-cbp.github.io/teaching/tutorials/protein/md.mdp", INPUTS_DIR)

DT_PS = 0.002
SIM_TIME_NS = 2.0
NSTEPS_FULL = int(round(SIM_TIME_NS * 1000 / DT_PS))
TEST_NSTEPS = 5000  # ~10 ps, just to validate the pipeline end-to-end

md_mdp_2ns_path = INPUTS_DIR / "md_2ns.mdp"
set_mdp_param(md_mdp_path, "nsteps", NSTEPS_FULL, out_path=md_mdp_2ns_path)

md_mdp_test_path = INPUTS_DIR / "md_test.mdp"
set_mdp_param(md_mdp_path, "nsteps", TEST_NSTEPS, out_path=md_mdp_test_path)

TEST_MODE = True  # set to False once the smoke test below has passed for all three force fields

print(f"Full production run: {NSTEPS_FULL} steps ({SIM_TIME_NS} ns)")
print(f"Smoke-test run:      {TEST_NSTEPS} steps ({TEST_NSTEPS * DT_PS} ps)")


In [ ]:
def run_production_md(name, workdir, mdp_path):
    shutil.copy(mdp_path, workdir / "md.mdp")
    run("gmx grompp -f md.mdp -c em.pdb -p topol.top -o md.tpr -maxwarn 2", cwd=workdir)
    run("gmx mdrun -v -s md.tpr -c md.pdb -deffnm md", cwd=workdir)
    return {
        "workdir": workdir,
        "performance": parse_performance(workdir / "md.log"),
        "atoms": parse_atom_counts(workdir / "topol.top"),
    }


In [ ]:
# Step 1: smoke test (fast) — validates all three force fields before the expensive full runs.
if TEST_MODE:
    for name, workdir in systems.items():
        run_production_md(name, workdir, md_mdp_test_path)
    print("\nSmoke test finished for all force fields — pipeline looks OK, proceed to the full run.")


In [ ]:
# Step 2: full 2 ns production runs (the expensive part: ~1.5 h per force field).
results = {}
for name, workdir in systems.items():
    results[name] = run_production_md(name, workdir, md_mdp_2ns_path)


**QUESTION:** How do the parts of energy minimization and MD simulation differ with reference to the energy landscape $V\left(\vec{r1},\vec{r2},...,\vec{r_N}\right)$?


##D) Analysis of the GROMACS simulations

Each `run_*` directory now contains the same files the original tutorial describes per run:
*   `traj_comp.xtc` - the trajectory (coordinates) to be used for analyses
*   `traj.trr` - coordinates, velocities and forces written at less frequency
*   `state.cpt` - a state of the trajectory to be used for a restart in case of a crash
*   `ener.edr` - energies
*   `md.log` - a LOG file of mdrun (used above for the performance numbers)
*   `md.pdb` - the final coordinates of the simulation

### 1) Exploratory visual inspection

For each force field, download `<workdir>/md.pdb` (final coordinates) and
`<workdir>/traj_comp.xtc` (trajectory) and open them in PyMOL as in the original tutorial
(`File → open → md.pdb`, then `load ~/PATH/traj_comp.xtc, md`) to sanity-check that the protein
and its surroundings look reasonable before trusting the quantitative analysis below. To view only
the protein, use:
```
hide all
show cartoon
show sticks, poly
```

### 2) Removal of rigid body translation and rotation

We want the properties we calculate to be invariant under rotation and translation, so we remove
the protein's rigid-body rotation/translation for each force field (least-squares fit on the
backbone, protein-only output) — this is what most GROMACS analysis tools below do internally, but
we also produce an explicit fitted trajectory for visualization, as in the original tutorial.


In [ ]:
for name, res in results.items():
    workdir = res["workdir"]
    run(
        "gmx trjconv -f traj_comp.xtc -s md.tpr -fit rot+trans -o protein_fit.xtc",
        input_text="Backbone\nProtein\n",
        cwd=workdir,
    )
    run(
        "gmx trjconv -f traj_comp.xtc -s md.tpr -fit rot+trans -o protein_0ns.pdb -dump 0",
        input_text="Backbone\nProtein\n",
        cwd=workdir,
    )
print("Fitted trajectories written to <workdir>/protein_fit.xtc (load together with protein_0ns.pdb in PyMOL).")


### 3) General outcome metrics (Question 1)

- Integration steps needed for the 2 ns run (same for all force fields, since `dt` and total time
  are fixed).
- Particle count: the **protein-atom** difference OPLS-AA vs. GROMOS (`protein_atoms_saved_vs_OPLS`)
  is the headline united-atom number and the clean answer to Q1's "how many particles less"
  (clarification Q2: protein is the primary comparison). The full-system count (`n_atoms_total`) is
  reported only for context and is **water-count-dependent** (each force field is solvated and
  neutralized independently; GRAPPA uses TIP3P vs. SPC), so it must not be quoted as the
  united-atom answer.
- System composition (`composition_df`, in the next cell): per-run water and ion counts, so any
  system-total or total-energy difference can be traced to solvent/ion differences rather than to
  the force field alone.
- Performance (ns/day), parsed from each `md.log`'s `Performance:` line (as suggested in the
  assignment), and the resulting speed-up relative to OPLS-AA.


In [ ]:
print(f"Integration steps for {SIM_TIME_NS} ns at dt = {DT_PS} ps: {NSTEPS_FULL}")

rows = []
for name, res in results.items():
    rows.append({
        "force_field": name,
        "n_atoms_protein": res["atoms"]["n_atoms_protein"],
        "n_atoms_total": res["atoms"]["n_atoms_total"],
        "ns_per_day": res["performance"]["ns_per_day"],
    })
outcome_df = pd.DataFrame(rows).set_index("force_field")

baseline_atoms = outcome_df.loc["OPLS-AA", "n_atoms_protein"]
baseline_perf = outcome_df.loc["OPLS-AA", "ns_per_day"]
outcome_df["protein_atoms_saved_vs_OPLS"] = baseline_atoms - outcome_df["n_atoms_protein"]
outcome_df["speedup_vs_OPLS"] = outcome_df["ns_per_day"] / baseline_perf

# Defensive (see parse_atom_counts): show which moleculetype name(s) were counted as protein per
# force field, so a GRAPPA/AMBER naming mismatch surfaces here instead of silently skewing counts.
for name, res in results.items():
    print(f"{name}: protein moleculetype(s) matched = "
          f"{res['atoms']['matched_protein_moltypes']} -> "
          f"{res['atoms']['n_atoms_protein']} protein atoms")

outcome_df


In [ ]:
# Per-run system composition (water + ion counts). The system-total particle count and the total
# potential energy both depend on how many waters/ions each force field's INDEPENDENT solvation +
# neutralization produced, so report this alongside Q1 to interpret those numbers honestly.
_WATER = {"SOL", "WAT", "HOH", "TIP3", "TIP3P", "SPC", "SOL_"}
_NA = {"NA", "NA+", "SOD"}
_CL = {"CL", "CL-", "CLA"}

comp_rows = []
for name, res in results.items():
    counts = res["atoms"]["molecule_counts"]
    n_water = sum(v for k, v in counts.items() if k.upper() in _WATER)
    n_na = sum(v for k, v in counts.items() if k.upper() in _NA)
    n_cl = sum(v for k, v in counts.items() if k.upper() in _CL)
    comp_rows.append({
        "force_field": name,
        "n_water": n_water,
        "n_Na": n_na,
        "n_Cl": n_cl,
        "n_atoms_protein": res["atoms"]["n_atoms_protein"],
        "n_atoms_total": res["atoms"]["n_atoms_total"],
    })
composition_df = pd.DataFrame(comp_rows).set_index("force_field")
composition_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
outcome_df["n_atoms_protein"].plot(kind="bar", ax=axes[0], color=["#4C72B0", "#55A868", "#C44E52"])
axes[0].set_ylabel("Protein atoms")
axes[0].set_title("Protein particle count")

outcome_df["ns_per_day"].plot(kind="bar", ax=axes[1], color=["#4C72B0", "#55A868", "#C44E52"])
axes[1].set_ylabel("Performance (ns/day)")
axes[1].set_title("Simulation performance")
plt.tight_layout()
plt.show()


### 4) RMSD & radius of gyration (Question 2)

For a more quantitative view of the protein's stability than visual inspection, we look at how
fast and how far the protein deviates from the starting structure (RMSD) and how its overall size
changes (radius of gyration) — computed for **two atom selections** (whole protein and
backbone/Cα) per clarification Q1, so we can directly discuss how sensitive the stability picture
is to that choice, and overlaid across the three force fields.

<font color=red size=+1> **QUESTION:** </font> Why is there an initial rise in the RMSD (in each
force field)?

<font color=red size=+1> **QUESTION:** </font> What can you say about the size and overall shape
of the protein based on the time-trace of the radius of gyration (e.g. fluctuations and drift),
and does that picture differ between force fields?


In [ ]:
RMS_SELECTIONS = {
    "protein": "Protein\nProtein\n",
    "backbone": "Backbone\nBackbone\n",
    "calpha": "C-alpha\nC-alpha\n",
}
GYRATE_SELECTIONS = {
    "protein": "Protein\n",
    "backbone": "Backbone\n",
}

rmsd_data = {}   # rmsd_data[selection][force_field] = (time_ns, rmsd_nm)
gyrate_data = {}

for sel, input_text in RMS_SELECTIONS.items():
    rmsd_data[sel] = {}
    for name, res in results.items():
        workdir = res["workdir"]
        out_file = f"rmsd_{sel}.xvg"
        run(f"gmx rms -s md.tpr -f traj_comp.xtc -o {out_file}", input_text=input_text, cwd=workdir)
        t, y = load_xvg(workdir / out_file, to_ns=True)
        rmsd_data[sel][name] = (t, y)

for sel, input_text in GYRATE_SELECTIONS.items():
    gyrate_data[sel] = {}
    for name, res in results.items():
        workdir = res["workdir"]
        out_file = f"gyrate_{sel}.xvg"
        run(f"gmx gyrate -s md.tpr -f traj_comp.xtc -o {out_file}", input_text=input_text, cwd=workdir)
        data = load_xvg(workdir / out_file, to_ns=True)
        t, rg = data[0], data[1]  # column 1 = Rg; columns 2-4 = principal components
        gyrate_data[sel][name] = (t, rg)


In [ ]:
def plot_overlay(ax, series_dict, xlabel, ylabel, title):
    for label, (x, y) in series_dict.items():
        ax.plot(x, y, label=label, linewidth=1.2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
plot_overlay(axes[0, 0], rmsd_data["protein"], "Time (ns)", "RMSD (nm)", "RMSD — whole protein")
plot_overlay(axes[0, 1], rmsd_data["backbone"], "Time (ns)", "RMSD (nm)", "RMSD — backbone")
plot_overlay(axes[1, 0], gyrate_data["protein"], "Time (ns)", "Rg (nm)", "Radius of gyration — whole protein")
plot_overlay(axes[1, 1], gyrate_data["backbone"], "Time (ns)", "Rg (nm)", "Radius of gyration — backbone")
plt.tight_layout()
plt.show()


In [ ]:
# Summary statistics (skip the first ~10% as equilibration when reporting mean/std)
def summarize(series_dict, skip_fraction=0.1):
    rows = []
    for label, (x, y) in series_dict.items():
        n_skip = int(len(y) * skip_fraction)
        rows.append({"force_field": label, "mean": y[n_skip:].mean(), "std": y[n_skip:].std()})
    return pd.DataFrame(rows).set_index("force_field")

stability_summary = pd.concat(
    {
        "RMSD_protein_nm": summarize(rmsd_data["protein"]),
        "RMSD_backbone_nm": summarize(rmsd_data["backbone"]),
        "RMSD_calpha_nm": summarize(rmsd_data["calpha"]),
        "Rg_protein_nm": summarize(gyrate_data["protein"]),
        "Rg_backbone_nm": summarize(gyrate_data["backbone"]),
    },
    axis=1,
)
stability_summary


### 5) RMSF (Question 3)

If we now wish to see if the fluctuations are equally distributed over the protein, or if some
residues are more flexible than others, we compute residue-wise RMSF on the backbone for each
force field and overlay the profiles. In the original (single-force-field, OPLS-AA) run, four
regions stood out as flexible — around residues 1, 11, 21 and 38 (loops, with the alpha-helix and
beta-sheet comparatively rigid). Here we additionally flag the residues where GROMOS/GRAPPA differ
most from the OPLS-AA baseline, to directly address whether the union-atom or ML force field
changes the flexibility pattern.

<font color=red size=+1> **QUESTION:** </font> Which is the most flexible residue, in each force
field — is it the same residue across all three?


In [ ]:
rmsf_data = {}
for name, res in results.items():
    workdir = res["workdir"]
    run(
        "gmx rmsf -s md.tpr -f traj_comp.xtc -oq bfac.pdb -res -o rmsf.xvg",
        input_text="Backbone\n",
        cwd=workdir,
    )
    resid, rmsf = load_xvg(workdir / "rmsf.xvg")
    rmsf_data[name] = (resid, rmsf)

fig, ax = plt.subplots(figsize=(9, 4))
plot_overlay(ax, rmsf_data, "Residue", "RMSF (nm)", "Per-residue flexibility")
plt.tight_layout()
plt.show()


In [ ]:
# Per-residue RMSF differences vs. OPLS-AA baseline, aligned by residue id via an OUTER MERGE
# (robust to residue-count/numbering differences between force fields, e.g. united-atom GROMOS or
# the GRAPPA/AMBER topology) instead of a hard equality assert that would crash on any mismatch.
rmsf_frames = {}
for name in ("OPLS-AA", "GROMOS", "GRAPPA"):
    resid, rmsf = rmsf_data[name]
    rmsf_frames[name] = pd.DataFrame(
        {"residue": np.asarray(resid, dtype=int), f"RMSF_{name}": rmsf}
    )

# Warn (do not crash) if the residue sets do not align perfectly across force fields.
resid_sets = {name: set(df["residue"]) for name, df in rmsf_frames.items()}
baseline_set = resid_sets["OPLS-AA"]
for name in ("GROMOS", "GRAPPA"):
    only_base = baseline_set - resid_sets[name]
    only_other = resid_sets[name] - baseline_set
    if only_base or only_other:
        print(f"WARNING: residue sets differ OPLS-AA vs {name}: "
              f"{len(only_base)} residues only in OPLS-AA, {len(only_other)} only in {name}. "
              f"delta_{name}_vs_OPLS is defined only where residues overlap (NaN elsewhere).")

rmsf_diff_df = rmsf_frames["OPLS-AA"]
for name in ("GROMOS", "GRAPPA"):
    rmsf_diff_df = rmsf_diff_df.merge(rmsf_frames[name], on="residue", how="outer")
rmsf_diff_df = rmsf_diff_df.sort_values("residue").reset_index(drop=True)
for name in ("GROMOS", "GRAPPA"):
    rmsf_diff_df[f"delta_{name}_vs_OPLS"] = (
        rmsf_diff_df[f"RMSF_{name}"] - rmsf_diff_df["RMSF_OPLS-AA"]
    )

# Residues with the largest absolute flexibility change, per force field (non-overlapping
# residues have NaN delta and are dropped from the ranking).
for name in ("GROMOS", "GRAPPA"):
    col = f"delta_{name}_vs_OPLS"
    valid = rmsf_diff_df.dropna(subset=[col])
    top = valid.reindex(valid[col].abs().sort_values(ascending=False).index).head(5)
    print(f"\nTop 5 residues with largest RMSF change, {name} vs OPLS-AA:")
    print(top[["residue", "RMSF_OPLS-AA", f"RMSF_{name}", col]].to_string(index=False))


To see where the flexible/differing residues are located in the protein: download each force
field's `bfac.pdb` and open it in PyMOL, show as spheres, then color by
`spectrum b-factors` (Color menu `C` → `spectrum` → `b-factors`). Red/orange regions are
relatively flexible, white regions relatively rigid.

**Further evaluating the flexibility hypothesis** (clarification Q7): beyond the RMSF trace,
cross-check candidate residues by (a) confirming in PyMOL that the affected regions are loops
rather than secondary-structure elements, (b) checking whether the same residues show elevated
RMSD contributions or SASA changes (Sections 4/7 below), and (c) noting that a single 2 ns
trajectory cannot rule out that an observed difference is statistical noise — a second,
independent run (different seed) would be the natural follow-up if time permitted
(clarification Q6).


### 6) Potential energy (Question 4)

The simulation not only yields information on the structural properties of the system, but also
on the energetics. We plot the potential energy (`ener.edr`, `Potential` term) for all three
force fields.

<font color=red size=+1> **QUESTION:** </font> Each force field's potential energy initially rises
rapidly after which it relaxes again — can you think of an explanation for this behaviour?

<font color=red size=+1> **QUESTION:** </font> Do you think the length of our simulation (2 ns) is
sufficient to provide a faithful picture of the protein's conformations at equilibrium?


In [ ]:
energy_data = {}
for name, res in results.items():
    workdir = res["workdir"]
    run("gmx energy -f ener.edr -o energy.xvg", input_text="Potential\n\n", cwd=workdir)
    t, e = load_xvg(workdir / "energy.xvg", to_ns=True)
    energy_data[name] = (t, e)

fig, ax = plt.subplots(figsize=(8, 4))
plot_overlay(ax, energy_data, "Time (ns)", "Potential energy (kJ/mol)", "Potential energy")
plt.tight_layout()
plt.show()

energy_summary = summarize(energy_data)
energy_summary


**Interpreting absolute potential energy across force fields**: the functional forms, reference
zero-points, and parametrization philosophies differ between OPLS-AA, GROMOS (united-atom, fewer
explicit interactions), and GRAPPA (ML-fitted terms), so the *absolute* energy values are not
directly comparable across force fields. What is meaningful is (a) whether each trajectory's
potential energy has stabilized/equilibrated by 2 ns, (b) the magnitude of fluctuations around the
mean, and (c) whether energy behavior correlates with the RMSD/Rg stability picture from Section 4.
Repeat the energy analysis for a few other energy terms (e.g. `Bond`, `Coulomb-SR`, `LJ-SR`) to get
an impression of their individual behavior per force field. The next cell adds a system-size-
**normalized** (per-atom) potential energy, which removes the gross dependence on how many water
molecules each run happened to contain and is a more defensible cross-force-field comparison than
raw total `Potential`. (Thermostat note for the report, clarification Q4: the Berendsen thermostat
is kept for strict consistency with the tutorial protocol; V-rescale would be the more rigorous
choice for canonical sampling in longer production runs.)


In [ ]:
# Additional, more defensible cross-force-field metric: potential energy normalized PER ATOM.
# Raw total Potential is not comparable across force fields (different functional forms, reference
# points, AND different system size / water count); dividing by the total number of atoms removes
# the gross system-size dependence. It is still only a rough comparison because the functional
# forms differ - use it to support the relaxation/fluctuation discussion, not as an absolute
# ranking of force fields.
# NOTE: a *truly* protein-restricted potential energy would require defining `energygrps` in
# md.mdp and re-running (or `gmx mdrun -rerun`); that is a workflow change we deliberately do NOT
# make here, to keep the protocol identical across the three force fields.
epot_per_atom = {}
for name, (t, e) in energy_data.items():
    n_total = outcome_df.loc[name, "n_atoms_total"]
    epot_per_atom[name] = (t, e / n_total)

fig, ax = plt.subplots(figsize=(8, 4))
plot_overlay(ax, epot_per_atom, "Time (ns)", "Potential energy per atom (kJ/mol/atom)",
             "Potential energy per atom (system-size-normalized)")
plt.tight_layout()
plt.show()

epot_per_atom_summary = summarize(epot_per_atom)
epot_per_atom_summary


### 7) Solvent accessible surface area (SASA) (Question 5)

Another important check concerns the behaviour of the protein surface. We compute total SASA plus
a hydrophobic/hydrophilic split (charge-based, as in the original tutorial), matching the scope
agreed in clarification Q8 (total SASA + qualitative hydrophobic-patch comparison, not an
exhaustive per-residue breakdown), overlaid across the three force fields.

<font color=red size=+1> **QUESTION:** </font> Is the total (solvent-accessible) surface constant,
in each force field? Are there any hydrophobic groups exposed to the solvent during the
simulation (hint: check the type of residues exposed to the solvent with PyMoL)? Do the three
force fields agree on this?


In [ ]:
sasa_data = {}       # sasa_data[force_field] = (time_ns, total, hydrophobic, hydrophilic)
for name, res in results.items():
    workdir = res["workdir"]
    run(
        "gmx sasa -s md.tpr -f traj_comp.xtc -surface 'Protein' "
        "-output '\"Hydrophobic\" group \"Protein\" and charge {-0.2 to 0.2}; "
        "\"Hydrophilic\" group \"Protein\" and not charge {-0.2 to 0.2}; "
        "\"Total\" group \"Protein\"' -o area.xvg",
        cwd=workdir,
    )
    data = load_xvg(workdir / "area.xvg", to_ns=True)
    t, total, hydrophobic, hydrophilic = data[0], data[1], data[2], data[3]
    sasa_data[name] = (t, total, hydrophobic, hydrophilic)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, name in zip(axes, sasa_data.keys()):
    t, total, hydrophobic, hydrophilic = sasa_data[name]
    ax.plot(t, hydrophobic, label="Hydrophobic")
    ax.plot(t, hydrophilic, label="Hydrophilic")
    ax.plot(t, total, label="Total", linestyle="--", color="gray")
    ax.set_title(name)
    ax.set_xlabel("Time (ns)")
    ax.legend()
axes[0].set_ylabel("Area (nm$^2$)")
plt.tight_layout()
plt.show()

sasa_rows = []
for name, (t, total, hydrophobic, hydrophilic) in sasa_data.items():
    n_skip = int(len(total) * 0.1)
    sasa_rows.append({
        "force_field": name,
        "SASA_total_mean": total[n_skip:].mean(),
        "SASA_hydrophobic_mean": hydrophobic[n_skip:].mean(),
        "SASA_hydrophilic_mean": hydrophilic[n_skip:].mean(),
        "hydrophobic_fraction": hydrophobic[n_skip:].mean() / total[n_skip:].mean(),
    })
sasa_summary = pd.DataFrame(sasa_rows).set_index("force_field")
sasa_summary


For qualitative follow-up (clarification Q8/Q10): load each force field's `protein_0ns.pdb` (or a
late-trajectory frame) in PyMOL, color by hydrophobicity, and visually flag any patches that are
exposed in one force field but buried in another — 1–2 representative snapshots are enough for the
report, not an exhaustive per-residue table.


### Consolidated summary table for the report

In [ ]:
report_summary = (
    outcome_df.join(stability_summary)
    .join(energy_summary.add_prefix("Epot_"))
    .join(epot_per_atom_summary.add_prefix("EpotPerAtom_"))
    .join(sasa_summary)
)
report_summary.to_csv("results_summary.csv")
print(report_summary.to_string())
print()
print(report_summary.round(3).to_latex())


## E) Discussion scaffolding — mapping results to the 7 assignment questions

This section is a **scaffold**, not filled-in answers: no simulation has actually been run in
this sandbox, so the numbers referenced below don't exist yet. Once the notebook has been run in
Colab, replace each bullet with the actual interpretation, citing the specific figure/table.

1. **General outcome** (Section D.3): report `NSTEPS_FULL`; report the protein-atom difference
   OPLS-AA vs. GROMOS from `outcome_df["protein_atoms_saved_vs_OPLS"]`; report the performance
   gain from `outcome_df["speedup_vs_OPLS"]`, and connect the two (fewer particles + fewer/cheaper
   nonbonded interactions ⇒ higher ns/day).
2. **Stability (RMSD, Rg)** (Section D.4): compare `stability_summary` across force fields and
   selections; explicitly discuss whether the whole-protein vs. backbone/Cα comparison changes
   the conclusion (clarification Q1) — e.g., solvent-exposed side chains typically inflate
   whole-protein RMSD relative to backbone.
3. **Flexibility (RMSF)** (Section D.5): name the regions with the largest `delta_*_vs_OPLS`,
   relate them to secondary structure (loops vs. helix/sheet, cf. the original tutorial's
   observation that loops around residues 1/11/21/38 are most flexible), and propose the PyMOL/
   replicate follow-up checks noted above.
4. **Potential energy** (Section D.6): describe the equilibration behavior and fluctuation
   magnitude per force field; explicitly caveat that absolute values aren't comparable across
   force fields with different functional forms.
5. **SASA / hydrophobic exposure** (Section D.7): report `sasa_summary["hydrophobic_fraction"]`
   per force field and note any force field with a distinctly different exposed hydrophobic
   fraction, backed by a PyMOL snapshot.
6. **Why check different force fields?** — conceptual answer, not tied to a specific plot: force
   fields are approximations with different parametrization data/philosophies (all-atom empirical
   vs. united-atom vs. ML-fitted); results (stability, dynamics, energetics) can be
   force-field-dependent, so cross-checking guards against over-interpreting artifacts of one
   parametrization, and is standard practice before drawing biological conclusions.
7. **How can ML improve force fields?** — conceptual answer informed by the GRAPPA comparison:
   ML-derived force fields (e.g., GRAPPA) can fit bonded/non-bonded parameters directly to
   quantum-chemical or high-level reference data across a broad chemical space, potentially
   capturing effects (e.g., conformation-dependent parameters, transferability across residue
   types) that fixed-functional-form classical force fields approximate more crudely — while
   trading off interpretability and requiring careful validation, which is exactly what Sections
   D.3–D.7 of this notebook do.


## F) Suggested report structure

The graded deliverable is a **single PDF, 2–6 pages**, answering the 7 questions above with
supporting data/plots (per `assignment2_description.pdf`). A page-budget-aware structure (also
written out in `docs/assignment/report_outline.tex`):

| Section | Content | Maps to | Budget |
|---|---|---|---|
| Header | Matrikel numbers, grade/pass-fail choice | — | few lines |
| 1. Setup summary | One paragraph: 3 force fields, 2 ns each, shared protocol (thermostat/constraints unchanged, ion neutralization consistent) | context | ~0.25 page |
| 2. General outcome | Step count, particle-count table, performance table/bar chart | Q1 | ~0.5 page |
| 3. Structural stability | RMSD + Rg overlay figure (whole-protein & backbone), summary table, discussion of atom-selection sensitivity | Q2 | ~1 page |
| 4. Flexibility | RMSF overlay figure, top-differing-residues table, 1 PyMOL snapshot (b-factor colored), explanation | Q3 | ~1 page |
| 5. Potential energy | Overlay figure, brief discussion (equilibration, fluctuations, no absolute cross-FF comparison) | Q4 | ~0.5 page |
| 6. SASA / hydrophobicity | Total + hydrophobic/hydrophilic figure, 1 PyMOL snapshot if a notable difference exists | Q5 | ~0.75 page |
| 7. Discussion: why compare force fields | Conceptual, 1 short paragraph | Q6 | ~0.25 page |
| 8. Discussion: ML force fields | Conceptual, grounded in the GRAPPA results above | Q7 | ~0.25 page |
| References | Tutorial, GRAPPA, force field papers | — | as needed |

This keeps the report within 6 pages by treating Sections 2/7/8 as compact (table/bar-chart or
short paragraph), reserving the most space (Sections 3–4) for the two questions that need the
richest supporting figures.


##References:
* Brini, Emiliano, Carlos Simmerling, and Ken Dill. **Protein Storytelling Through Physics.** [Science 370:6520 (2020)](https://doi.org/10.1126/science.aaz3041).

* Ron O. Dror, Robert M. Dirks, J.P. Grossman, Huafeng Xu, and David E. Shaw. **Biomolecular Simulation: A Computational Microscope for Molecular Biology.** [Annual Review of Biophysics. 41:429-452 (2012).](https://doi.org/10.1146/annurev-biophys-042910-155245)

* M. Karplus and A. McCammon.  **Molecular Dynamics simulations of
biomolecules.** [Nature structural biology 9: 646-652 (2002).](http://www.nature.com/nsmb/journal/v9/n9/full/nsb0902-646.html)

* D.C. Rapaport. **The Art of Molecular Dynamics Simulations - 2nd edn**.Cambridge University Press. (2004).

* William L. Jorgensen, David S. Maxwell, and Julian Tirado-Rives. **Development and Testing of the OPLS All-Atom Force Field on Conformational Energetics and Properties of Organic Liquids.** [J. Am. Chem. Soc. 118:11225-11236 (1996).](https://doi.org/10.1021/ja9621760)

* Christine Oostenbrink, Alessandra Villa, Alan E. Mark, and Wilfred F. van Gunsteren. **A biomolecular force field based on the free enthalpy of hydration and solvation: The GROMOS force-field parameter sets 53A5 and 53A6.** [J. Comput. Chem. 25:1656-1676 (2004).](https://doi.org/10.1002/jcc.20090)

* GRAPPA (machine-learned force field): add the specific citation/course reference used in this
  course's materials (`docs/course_materials/`) once confirmed — not fabricated here to avoid citing
  the wrong paper.
